In [3]:
import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
import warnings
warnings.filterwarnings('ignore')


# ==========================================================
# Step 1: Load Data
# ==========================================================
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

test_ids = test["PassengerId"]
y = train["Transported"].astype(int)

# Combine train + test
full = pd.concat(
    [train.drop("Transported", axis=1), test],
    axis=0
).reset_index(drop=True)

# ==========================================================
# Step 2: Feature Engineering
# ==========================================================

# Group Features
full["GroupID"] = full["PassengerId"].str.split("_").str[0]
full["GroupSize"] = full.groupby("GroupID")["GroupID"].transform("count")

# Cabin Features
full[["Deck", "CabinNum", "Side"]] = full["Cabin"].str.split("/", expand=True)
full["CabinNum"] = pd.to_numeric(full["CabinNum"], errors="coerce")

# Spending Features
spend_cols = [
    "RoomService",
    "FoodCourt",
    "ShoppingMall",
    "Spa",
    "VRDeck"
]

for col in spend_cols:
    full[col] = full[col].fillna(0)

full["TotalSpent"] = full[spend_cols].sum(axis=1)

# Age
full["Age"] = full["Age"].fillna(full["Age"].median())

# Boolean Features
full["CryoSleep"] = full["CryoSleep"].fillna(False).astype(int)
full["VIP"] = full["VIP"].fillna(False).astype(int)

# ==========================================================
# Step 3: Fill Missing Values
# ==========================================================
for col in full.columns:

    if full[col].dtype == "object":
        full[col] = full[col].fillna(full[col].mode()[0])

    else:
        full[col] = full[col].fillna(full[col].median())

# ==========================================================
# Step 4: Drop Weak Columns
# ==========================================================
full.drop(
    ["PassengerId", "Cabin", "Name", "GroupID"],
    axis=1,
    inplace=True
)

# ==========================================================
# Step 5: One-Hot Encoding
# ==========================================================
full = pd.get_dummies(
    full,
    columns=["HomePlanet", "Destination", "Deck", "Side"],
    drop_first=True
)

# ==========================================================
# Step 6: Split Train / Test
# ==========================================================
X = full.iloc[:len(train), :]
X_test = full.iloc[len(train):, :]

# ==========================================================
# Step 7: Breakthrough Random Forest
# ==========================================================
rf = RandomForestClassifier(
    n_estimators=2600,
    max_depth=12,
    min_samples_leaf=2,
    min_samples_split=12,
    max_features="log2",
    criterion="entropy",
    bootstrap=True,
    random_state=42,
    n_jobs=-1
)

# ==========================================================
# Step 8: Print Accuracy
# ==========================================================
scores = cross_val_score(
    rf,
    X,
    y,
    cv=5,
    scoring="accuracy"
)

print("CV Scores:", scores)
print("Mean Accuracy:", scores.mean())

# ==========================================================
# Step 9: Train Full Model
# ==========================================================
rf.fit(X, y)

# ==========================================================
# Step 10: Predict Test Set
# ==========================================================
pred = rf.predict(X_test)

# ==========================================================
# Step 11: Create Submission
# ==========================================================
submission = pd.DataFrame({
    "PassengerId": test_ids,
    "Transported": pred.astype(bool)
})

submission.to_csv("submission.csv", index=False)

print("submission.csv created successfully!")
print("Upload to Kaggle now.")

CV Scores: [0.77055779 0.7786084  0.80563542 0.83084005 0.80207135]
Mean Accuracy: 0.7975426005051646
submission.csv created successfully!
Upload to Kaggle now.
